Codice per il download del database NVD contenente CVE, CWE, CVSS, CPE
inserire in fondo api key nvd

In [ ]:
import requests
import csv
import time
from datetime import datetime
import json

class NVDDownloader:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
        self.headers = {
            "apiKey": api_key
        }

        self.rate_limit_delay = 0.6  

    def get_cves(self, start_index=0, results_per_page=2000):
        params = {
            "startIndex": start_index,
            "resultsPerPage": results_per_page
        }

        try:
            response = requests.get(
                self.base_url,
                headers=self.headers,
                params=params,
                timeout=30
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Errore nella richiesta: {e}")
            return None

    def extract_cvss_data(self, metrics):
        cvss_score = 'N/A'
        cvss_vector = 'N/A'
        cvss_version = 'N/A'


        if 'cvssMetricV40' in metrics and metrics['cvssMetricV40']:
            cvss_data = metrics['cvssMetricV40'][0].get('cvssData', {})
            cvss_score = cvss_data.get('baseScore', 'N/A')
            cvss_vector = cvss_data.get('vectorString', 'N/A')
            cvss_version = 'v4.0'
        elif 'cvssMetricV31' in metrics and metrics['cvssMetricV31']:
            cvss_data = metrics['cvssMetricV31'][0].get('cvssData', {})
            cvss_score = cvss_data.get('baseScore', 'N/A')
            cvss_vector = cvss_data.get('vectorString', 'N/A')
            cvss_version = 'v3.1'
        elif 'cvssMetricV30' in metrics and metrics['cvssMetricV30']:
            cvss_data = metrics['cvssMetricV30'][0].get('cvssData', {})
            cvss_score = cvss_data.get('baseScore', 'N/A')
            cvss_vector = cvss_data.get('vectorString', 'N/A')
            cvss_version = 'v3.0'
        elif 'cvssMetricV2' in metrics and metrics['cvssMetricV2']:
            cvss_data = metrics['cvssMetricV2'][0].get('cvssData', {})
            cvss_score = cvss_data.get('baseScore', 'N/A')
            cvss_vector = cvss_data.get('vectorString', 'N/A')
            cvss_version = 'v2.0'

        return cvss_score, cvss_vector, cvss_version

    def extract_cpe_data(self, configurations):
        cpe_list = []

        for config in configurations:
            nodes = config.get('nodes', [])
            for node in nodes:
                cpe_match = node.get('cpeMatch', [])
                for cpe in cpe_match:
                    if cpe.get('vulnerable', False):
                        cpe_uri = cpe.get('criteria', '')
                        if cpe_uri and cpe_uri not in cpe_list:
                            cpe_list.append(cpe_uri)

        return '; '.join(cpe_list) if cpe_list else 'N/A'

    def extract_cve_data(self, vulnerability):
        cve = vulnerability.get('cve', {})

        cve_id = cve.get('id', 'N/A')

        # Descrizione
        descriptions = cve.get('descriptions', [])
        description = 'N/A'
        for desc in descriptions:
            if desc.get('lang') == 'en':
                description = desc.get('value', 'N/A')
                break

        # CWE
        weaknesses = cve.get('weaknesses', [])
        cwe_list = []
        for weakness in weaknesses:
            for desc in weakness.get('description', []):
                cwe_value = desc.get('value', '')
                if cwe_value and cwe_value not in cwe_list:
                    cwe_list.append(cwe_value)

        cwe_string = ', '.join(cwe_list) if cwe_list else 'N/A'

        # CVSS 
        metrics = cve.get('metrics', {})
        cvss_score, cvss_vector, cvss_version = self.extract_cvss_data(metrics)

        # CPE
        configurations = cve.get('configurations', [])
        cpe_string = self.extract_cpe_data(configurations)

        return {
            'cve_id': cve_id,
            'description': description,
            'cwe': cwe_string,
            'cvss_score': cvss_score,
            'cvss_vector': cvss_vector,
            'cvss_version': cvss_version,
            'cpe': cpe_string
        }

    def download_all_cves(self, output_file='nvd_cves.csv'):
        print("download delle CVE dal database NVD")
        print(f"File di output: {output_file}")

        print("\nnumero totale di CVE")
        first_response = self.get_cves(start_index=0, results_per_page=1)

        if not first_response:
            print("Errore nell'ottenere i dati iniziali")
            return

        total_results = first_response.get('totalResults', 0)
        print(f"Totale CVE da scaricare: {total_results}")

        with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['cve_id', 'description', 'cwe', 'cvss_score', 'cvss_vector', 'cvss_version', 'cpe']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

            results_per_page = 2000
            start_index = 0
            total_downloaded = 0

            while start_index < total_results:
                print(f"\nScaricamento CVE {start_index} - {min(start_index + results_per_page, total_results)} di {total_results}")

                response = self.get_cves(start_index, results_per_page)

                if not response:
                    print(f"Errore nel download del batch a startIndex={start_index}")
                    time.sleep(30)
                    continue

                vulnerabilities = response.get('vulnerabilities', [])

                # Processa ogni CVE
                for vuln in vulnerabilities:
                    cve_data = self.extract_cve_data(vuln)
                    writer.writerow(cve_data)
                    total_downloaded += 1

                print(f"Scaricate: {total_downloaded} CVE")

                start_index += results_per_page

                if start_index < total_results:
                    time.sleep(self.rate_limit_delay)

        print(f"\nDownload completato")
        print(f"Totale CVE scaricate: {total_downloaded}")
        print(f"File salvato: {output_file}")

def main():
    API_KEY = "" #inserire api key

    OUTPUT_FILE = f"nvd_cves_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

    downloader = NVDDownloader(API_KEY)
    downloader.download_all_cves(OUTPUT_FILE)

if __name__ == "__main__":
    main()

Inizio download delle CVE dal database NVD...
File di output: nvd_cves_20260109_143137.csv

Ottenimento del numero totale di CVE...
Totale CVE da scaricare: 326943

Scaricamento CVE 0 - 2000 di 326943
Scaricate: 2000 CVE

Scaricamento CVE 2000 - 4000 di 326943
Scaricate: 4000 CVE

Scaricamento CVE 4000 - 6000 di 326943
Scaricate: 6000 CVE

Scaricamento CVE 6000 - 8000 di 326943
Scaricate: 8000 CVE

Scaricamento CVE 8000 - 10000 di 326943
Scaricate: 10000 CVE

Scaricamento CVE 10000 - 12000 di 326943
Scaricate: 12000 CVE

Scaricamento CVE 12000 - 14000 di 326943
Scaricate: 14000 CVE

Scaricamento CVE 14000 - 16000 di 326943
Scaricate: 16000 CVE

Scaricamento CVE 16000 - 18000 di 326943
Scaricate: 18000 CVE

Scaricamento CVE 18000 - 20000 di 326943
Scaricate: 20000 CVE

Scaricamento CVE 20000 - 22000 di 326943
Scaricate: 22000 CVE

Scaricamento CVE 22000 - 24000 di 326943
Scaricate: 24000 CVE

Scaricamento CVE 24000 - 26000 di 326943
Scaricate: 26000 CVE

Scaricamento CVE 26000 - 28000 d